In [17]:
import torch as tr
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import tiktoken as ttk
import numpy as np
import importlib
import sys
import os

# Get the absolute path of the directory you want to import from
# module_dir = os.path.abspath('/Users/anujvora/Library/CloudStorage/Dropbox/Online courses/LLM_build/LLM_scratch/Experiments-with-GPT/GPT_architecture')
module_dir = [os.path.relpath('/Experiments-with-GPT/GPT_architecture'), os.path.relpath('/Experiments-with-GPT/data')]

# Add the directory to the system path
for direc in module_dir:
    if direc not in sys.path:
        sys.path.append(direc)

print(sys.path)
# Now you can import the module as if it were in the current directory

import GPT_modules as gpt
import matplotlib.pyplot as pl

# importlib.reload(gpt) 



GPT_CONFIG_124M = {
"vocab_size": 50257, # Vocabulary size
"context_length": 256, # Context length
"emb_dim": 768, # Embedding dimension
"n_heads": 12, # Number of attention heads
"n_layers": 12, # Number of layers
"drop_rate": 0.1, # Dropout rate
"qkv_bias": False # Query-Key-Value bias
}


tokenizer = ttk.get_encoding("gpt2")
gpt_model = gpt.GPTModule(GPT_CONFIG_124M)
gpt_model.eval()

['/Library/Frameworks/Python.framework/Versions/3.10/lib/python310.zip', '/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10', '/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/lib-dynload', '', '/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages', '/Users/anujvora/Library/CloudStorage/Dropbox/Online courses/LLM_build/LLM_scratch/Experiments-with-GPT', '/Users/anujvora/Library/CloudStorage/Dropbox/Online courses/LLM_build/LLM_scratch/Experiments-with-GPT\\GPT_modules', '/Users/anujvora/Library/CloudStorage/Dropbox/Online courses/LLM_build/LLM_scratch/\\Experiments-with-GPT\\GPT_modules', '/Experiments-with-GPT', '/Experiments-with-GPT/GPT_architecture', '/Users/anujvora/Library/CloudStorage/Dropbox/Online courses/LLM_build/LLM_scratch/Experiments-with-GPT/GPT_architecture', '../../../../../../../../Experiments-with-GPT/GPT_architecture', '../../../../../../../../Experiments-with-GPT/data']


GPTModule(
  (Token_Embedding): Embedding(50257, 768)
  (Position_Embedding): Embedding(256, 768)
  (Dropout_bef_Transform): Dropout(p=0.1, inplace=False)
  (Transformer_block): Sequential(
    (0): Transformer(
      (Layer_norm1): LayerNormalization()
      (Layer_norm2): LayerNormalization()
      (Multihead_attn): MultiHeadAttention(
        (W_query): Linear(in_features=768, out_features=768, bias=False)
        (W_key): Linear(in_features=768, out_features=768, bias=False)
        (W_value): Linear(in_features=768, out_features=768, bias=False)
        (dropout): Dropout(p=0.1, inplace=False)
        (out_proj): Linear(in_features=768, out_features=768, bias=True)
      )
      (FFN): FFN(
        (layers): Sequential(
          (0): Linear(in_features=768, out_features=3072, bias=True)
          (1): GELU(approximate='none')
          (2): Linear(in_features=3072, out_features=768, bias=True)
        )
      )
      (Dropout): Dropout(p=0.1, inplace=False)
    )
    (1): Transfo

In [39]:
def text_to_token(text,tokenizer):
    tokens = tokenizer.encode(text)
    return tr.tensor((tokens)).unsqueeze(0)

def token_to_text(tokens,tokenizer):
    tokens_list = tokens.squeeze(0).tolist()
    return tokenizer.decode(tokens_list)

# predicted_token_list = input_tokens.squeeze(0).tolist()
# tokenizer.decode(predicted_token_list)
def print_sample_text(input_sentence,tokenizer,gpt_model):
    max_length_text = 20
# input_sentence = "Topic"
# # # input_tokens = tr.tensor()
# new_input = tokenizer.encode(input_sentence)

# # input_tokens.shape
    input_tokens = text_to_token(input_sentence,tokenizer)
    for _ in range(max_length_text):
        output = nn.functional.softmax(gpt_model(input_tokens)[:,-1],dim=-1)
        # print(output.shape)
        new_input = tr.argmax(output)
        # print(input_tokens,new_input.view(1,1))
        input_tokens = tr.cat((input_tokens,new_input.view(1,1)),dim=1)
        # print(new_input,input_tokens)

    print(token_to_text(input_tokens,tokenizer))


In [28]:
nn.functional.cross_entropy(output,tr.tensor([300])), output.shape

(tensor(10.8250, grad_fn=<NllLossBackward0>), torch.Size([1, 50257]))

In [ ]:
# print(sys.path)
# open('Experiments-with-GPT/data/the_verdict.txt','r')

['/Library/Frameworks/Python.framework/Versions/3.10/lib/python310.zip', '/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10', '/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/lib-dynload', '', '/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages', '/Users/anujvora/Library/CloudStorage/Dropbox/Online courses/LLM_build/LLM_scratch/Experiments-with-GPT', '/Users/anujvora/Library/CloudStorage/Dropbox/Online courses/LLM_build/LLM_scratch/Experiments-with-GPT\\GPT_modules', '/Users/anujvora/Library/CloudStorage/Dropbox/Online courses/LLM_build/LLM_scratch/\\Experiments-with-GPT\\GPT_modules', '/Experiments-with-GPT', '/Experiments-with-GPT/GPT_architecture', '/Users/anujvora/Library/CloudStorage/Dropbox/Online courses/LLM_build/LLM_scratch/Experiments-with-GPT/GPT_architecture', '../../../../../../../../Experiments-with-GPT/GPT_architecture', '../../../../../../../../Experiments-with-GPT/data']


<_io.TextIOWrapper name='Experiments-with-GPT/data/the_verdict.txt' mode='r' encoding='UTF-8'>

In [ ]:
with open('/data/the_verdict.txt','r',encoding='utf-8') as file:
    raw_text = file.read()
    
print('Total characters:', len(raw_text))
print(raw_text[:99])

train_ind = 15000
train_inp_tar_dataset = gpt.GPTDataset(raw_text[:train_ind],tokenizer,GPT_CONFIG_124M["context_length"],stride=GPT_CONFIG_124M["context_length"])
test_inp_tar_dataset = gpt.GPTDataset(raw_text[train_ind:],tokenizer,GPT_CONFIG_124M["context_length"],stride=GPT_CONFIG_124M["context_length"])
# print(f"dataset size:{len(dataset)}")

Total characters: 20479
I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no 


In [28]:
train_dataset = DataLoader(train_inp_tar_dataset,batch_size=2,shuffle=False)
test_dataset = DataLoader(test_inp_tar_dataset,batch_size=2,shuffle=False)

In [6]:
for t in train_dataset:
    print(t[0].shape)

torch.Size([2, 256])
torch.Size([2, 256])
torch.Size([2, 256])
torch.Size([2, 256])
torch.Size([2, 256])
torch.Size([2, 256])
torch.Size([2, 256])


In [ ]:
loss_crit = nn.CrossEntropyLoss()
def calc_loss_batch(output,tar_batch):
    # output = model(inp_batch)
    # print(output.shape)
    out_flat = output.flatten(0,1)
    tar_flat = tar_batch.flatten()
    # print(out_flat.shape,tar_flat.shape)
    # print(nn.functional.softmax(out_flat,dim=-1))
    # print(loss_crit(nn.functional.softmax(out_flat,dim=-1),tar_flat))
    return nn.functional.cross_entropy(out_flat,tar_flat,reduction='mean')

# for inp_batch, tar_batch in train_dataset:
#     output = gpt_model(inp_batch)
#     print(calc_loss_batch(output,tar_batch))

# size(tar_batch)

In [30]:
n_epoch = 5

tr.manual_seed(123)
gpt_model = gpt.GPTModule(GPT_CONFIG_124M)
optimizer = tr.optim.Adam(gpt_model.parameters(),lr=1e-4)##,weight_decay=0.1)
# loss_criteria = nn.CrossEntropyLoss()                                                                                              
loss_epoch = []
for epoch in range(n_epoch):
    gpt_model.train()
    loss_batch = 0
    for i,(input,target) in enumerate(train_dataset):
        optimizer.zero_grad()
        # print(token_to_text(input[0],tokenizer),'\n',token_to_text(target[0],tokenizer))
        output_batch = gpt_model(input)
        
        loss = calc_loss_batch(output_batch,target)
        loss.backward()
        optimizer.step()
        loss_batch += loss
    print_sample_text("Every effort moves you",tokenizer,gpt_model)

    loss_epoch.append(loss_batch)

loss_epoch

Every effort moves you,,,,,,,,,,,,,,,,,,,,
Every effort moves you,,,,,,,,,,,,,,,,,,,,
Every effort moves you, the, the, the,,,, the,, the,, the,,,
Every effort moves you, the.

















Every effort moves you.




".


".

".





[tensor(70.8497, grad_fn=<AddBackward0>),
 tensor(61.2778, grad_fn=<AddBackward0>),
 tensor(55.7815, grad_fn=<AddBackward0>),
 tensor(50.1150, grad_fn=<AddBackward0>),
 tensor(44.7396, grad_fn=<AddBackward0>)]

In [40]:
def generate(input_sentence,tokenizer,gpt_model,context_size,temperature,max_length_text,top_k):
    input_tokens = text_to_token(input_sentence,tokenizer)[:,-context_size:]

    for _ in range(max_length_text):
        with tr.no_grad():
            output = gpt_model(input_tokens)[:,-1] #
        if top_k:
            top_k_vals, top_k_ind = tr.topk(output,top_k)
            # print(output.shape,top_k_vals)
            new_output = tr.where(condition=output < top_k_vals[:,-1],
                                  input= tr.tensor(float("-inf")),
                                  other=output)
            # print(new_output[:,top_k_ind])
            temp_scaled_normalized = nn.functional.softmax(new_output/temperature,dim=-1)
            # print(temp_scaled_normalized[:,top_k_ind])
            new_input = tr.multinomial(temp_scaled_normalized,num_samples=1)
        else:
            new_input = tr.argmax(output,keepdim=True)
        input_tokens = tr.cat((input_tokens,new_input),dim=1)

    return token_to_text(input_tokens,tokenizer)

input_sentence = "Every one"
# top_k = 3
text_gen = generate(input_sentence,tokenizer,gpt_model,GPT_CONFIG_124M["context_length"],temperature=2,max_length_text=10,top_k=3)
print(text_gen)

Every one who does not believe in God, and who does


In [37]:
tr.save(gpt_model.state_dict(),'LLM_model1.pth')

In [38]:
new_model = gpt.GPTModule(GPT_CONFIG_124M)
new_model.load_state_dict(tr.load("LLM_model1.pth"))

<All keys matched successfully>

In [39]:
tr.save({"model_state_dict": gpt_model.state_dict(),
         "optimizer_state_dict": optimizer.state_dict()},"LLM_model_optimizer.pth")

In [43]:
ckpt = tr.load("LLM_model_optimizer.pth")
new_model.load_state_dict(ckpt["model_state_dict"])
optimizer.load_state_dict(ckpt["optimizer_state_dict"])

In [3]:
import urllib.request
url = (
"https://raw.githubusercontent.com/rasbt/"
"LLMs-from-scratch/main/ch05/"
"01_main-chapter-code/gpt_download.py"
)
filename = url.split('/')[-1]
urllib.request.urlretrieve(url, filename)

('gpt_download.py', <http.client.HTTPMessage at 0x13fbf4e50>)

In [23]:
from gpt_download import download_and_load_gpt2
settings, params = download_and_load_gpt2(model_size="124M",models_dir="gpt2")

File already exists and is up-to-date: gpt2/124M/checkpoint
File already exists and is up-to-date: gpt2/124M/encoder.json
File already exists and is up-to-date: gpt2/124M/hparams.json
File already exists and is up-to-date: gpt2/124M/model.ckpt.data-00000-of-00001
File already exists and is up-to-date: gpt2/124M/model.ckpt.index
File already exists and is up-to-date: gpt2/124M/model.ckpt.meta
File already exists and is up-to-date: gpt2/124M/vocab.bpe


In [24]:
print(settings, params.keys())
print(len(params['blocks']),params['blocks'][0].keys())
print(params['blocks'][0]['attn'].keys(), params['blocks'][0].keys())
print(params['blocks'][0]['ln_1'].keys())

{'n_vocab': 50257, 'n_ctx': 1024, 'n_embd': 768, 'n_head': 12, 'n_layer': 12} dict_keys(['blocks', 'b', 'g', 'wpe', 'wte'])
12 dict_keys(['attn', 'ln_1', 'ln_2', 'mlp'])
dict_keys(['c_attn', 'c_proj']) dict_keys(['attn', 'ln_1', 'ln_2', 'mlp'])
dict_keys(['b', 'g'])


In [25]:
model_configs = {"gpt2-small": {"emb_dim":768,"n_layers": 12,"n_heads": 12}}
NEW_CONFIG = GPT_CONFIG_124M.copy()
NEW_CONFIG.update(model_configs["gpt2-small"])
NEW_CONFIG.update({"context_size":1024, "qkv_bias":True})

gpt_model = gpt.GPTModule(NEW_CONFIG)
gpt_model.eval()

GPTModule(
  (Token_Embedding): Embedding(50257, 768)
  (Position_Embedding): Embedding(256, 768)
  (Dropout_bef_Transform): Dropout(p=0.1, inplace=False)
  (Transformer_block): Sequential(
    (0): Transformer(
      (Layer_norm1): LayerNormalization()
      (Layer_norm2): LayerNormalization()
      (Multihead_attn): MultiHeadAttention(
        (W_query): Linear(in_features=768, out_features=768, bias=True)
        (W_key): Linear(in_features=768, out_features=768, bias=True)
        (W_value): Linear(in_features=768, out_features=768, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
        (out_proj): Linear(in_features=768, out_features=768, bias=True)
      )
      (FFN): FFN(
        (layers): Sequential(
          (0): Linear(in_features=768, out_features=3072, bias=True)
          (1): GELU(approximate='none')
          (2): Linear(in_features=3072, out_features=768, bias=True)
        )
      )
      (Dropout): Dropout(p=0.1, inplace=False)
    )
    (1): Transforme

In [34]:
def assign(left_val,right_val):
    if left_val.shape != right_val.shape:
        ValueError(f"Left {left_val.shape} and right {right_val.shape} shapes do not match")
    return tr.nn.Parameter(tr.tensor(right_val))

def load_weights_into_model(gpt_model,params):
    gpt_model.Position_Embedding.weight = assign(gpt_model.Position_Embedding.weight,params["wpe"])
    gpt_model.Token_Embedding.weight = assign(gpt_model.Token_Embedding.weight,params["wte"])

    for b in range(len(params["blocks"])):
        ## attention block weights, biases, and projections
        q_w, k_w, v_w = np.split(params["blocks"][b]["attn"]["c_attn"]["w"], 3, axis=-1)
        gpt_model.Transformer_block[b].Multihead_attn.W_query.weight = assign(gpt_model.Transformer_block[b].Multihead_attn.W_query.weight, q_w.T)
        gpt_model.Transformer_block[b].Multihead_attn.W_key.weight = assign(gpt_model.Transformer_block[b].Multihead_attn.W_key.weight, k_w.T)
        gpt_model.Transformer_block[b].Multihead_attn.W_value.weight = assign(gpt_model.Transformer_block[b].Multihead_attn.W_value.weight, v_w.T)

        q_b, k_b, v_b = np.split(params["blocks"][b]["attn"]["c_attn"]["b"], 3, axis=-1)
        gpt_model.Transformer_block[b].Multihead_attn.W_query.bias = assign(gpt_model.Transformer_block[b].Multihead_attn.W_query.bias, q_b)
        gpt_model.Transformer_block[b].Multihead_attn.W_key.bias = assign(gpt_model.Transformer_block[b].Multihead_attn.W_key.bias, k_b)
        gpt_model.Transformer_block[b].Multihead_attn.W_value.bias = assign(gpt_model.Transformer_block[b].Multihead_attn.W_value.bias, v_b)

        gpt_model.Transformer_block[b].Multihead_attn.out_proj.weight = assign(gpt_model.Transformer_block[b].Multihead_attn.out_proj.weight, params["blocks"][b]["attn"]["c_proj"]["w"].T)
        gpt_model.Transformer_block[b].Multihead_attn.out_proj.bias = assign(gpt_model.Transformer_block[b].Multihead_attn.out_proj.bias, params["blocks"][b]["attn"]["c_proj"]["b"])

        ## feedforward layers
        gpt_model.Transformer_block[b].FFN.layers[0].weight = assign(gpt_model.Transformer_block[b].FFN.layers[0].weight, params["blocks"][b]["mlp"]["c_fc"]["w"].T)
        gpt_model.Transformer_block[b].FFN.layers[0].bias = assign(gpt_model.Transformer_block[b].FFN.layers[0].bias, params["blocks"][b]["mlp"]["c_fc"]["b"])

        gpt_model.Transformer_block[b].FFN.layers[2].weight = assign(gpt_model.Transformer_block[b].FFN.layers[2].weight, params["blocks"][b]["mlp"]["c_proj"]["w"].T)
        gpt_model.Transformer_block[b].FFN.layers[2].bias = assign(gpt_model.Transformer_block[b].FFN.layers[2].bias, params["blocks"][b]["mlp"]["c_proj"]["b"])

        ## layer normalizations
        gpt_model.Transformer_block[b].Layer_norm1.scale = assign(gpt_model.Transformer_block[b].Layer_norm1.scale, params["blocks"][b]["ln_1"]["g"])
        gpt_model.Transformer_block[b].Layer_norm1.shift = assign(gpt_model.Transformer_block[b].Layer_norm1.shift, params["blocks"][b]["ln_1"]["b"])
        gpt_model.Transformer_block[b].Layer_norm2.scale = assign(gpt_model.Transformer_block[b].Layer_norm2.scale, params["blocks"][b]["ln_2"]["g"])
        gpt_model.Transformer_block[b].Layer_norm2.shift = assign(gpt_model.Transformer_block[b].Layer_norm2.shift, params["blocks"][b]["ln_2"]["b"])

    ## final layer norm
    gpt_model.Final_Layer_Norm.scale = assign(gpt_model.Final_Layer_Norm.scale, params["g"])
    gpt_model.Final_Layer_Norm.shift = assign(gpt_model.Final_Layer_Norm.shift, params["b"])
    gpt_model.Final_output.weight = assign(gpt_model.Final_output.weight, params["wte"])

load_weights_into_model(gpt_model,params)

In [95]:
input_sentence = "Make hay"
context_size, temperature = NEW_CONFIG["context_length"],1.5
max_length_text, top_k = 20,30
generated_string = generate(input_sentence,tokenizer,gpt_model,context_size,temperature,max_length_text,top_k)
print(generated_string)

Make hay to go for two-and-a-half hours during summertime on your way around town.
